In [2]:
"""
Graph-structure fidelity metrics for Berka-style transaction tables
(real vs. ClavaDDPM-synthetic).

Usage
-----
    real_trans = pd.read_csv(REAL_TRANS_PATH)
    synthetic_trans = pd.read_csv(SYNTHETIC_TRANS_PATH)
    results = evaluate(real_trans, synthetic_trans)

Only needs the `trans` table as ClavaDDPM outputs it (account_id, date, amount,
type, operation, k_symbol, bank, account). No upstream changes are required;
if the synthetic counterparty identities are not reused, the metrics simply
show that gap.

Terminology: a "counterparty" is identified by "bank:account" of the other side of a
transaction (rows whose counterparty account is a missing value have none). An "edge" is a
transaction with a counterparty; a "pair" is an (account, counterparty) combination;
a pair with several transactions is a "repeated" pair (recurring payment).

Metric groups (all isomorphism-invariant: no node matching is needed, and they
work with or without communities). Output metric names use the abbreviation "cp" for
counterparty:

  presence          counterparty-present rate, per-account share, rate by type/operation/k_symbol
  account           per-account degree / repeat / concentration features
  pair              pair multiplicity (transactions per account-counterparty pair)
  counterparty      counterparty in-degree (distinct accounts) and volume, hub concentration
  edge_consistency  attribute stability inside repeated pairs
  structure         giant/dyad components; wedges + butterflies (+ null z) only if the real
                    data has shared counterparties (min_shared_counterparty_share_for_motifs)
  bank              coarse account x bank graph: bank mix, banks per account, bank co-usage
  temporal          inter-event gaps per pair, gap regularity, monthly recurrence
  coupling          amount/type/operation/k_symbol/bank conditioned on pair multiplicity
                    and counterparty in-degree
  c2st              classifier two-sample test on account-level graph features

Every distance is reported next to a noise floor from real-vs-real (random split of
the real accounts into two halves). ratio = distance / floor; ratio <= GAP_RATIO_THRESHOLD
-> "ok" (heuristic). If the number of accounts differs by more than 5%, the larger side is
subsampled by accounts so scale-dependent metrics are comparable.

Repeat-dependent metrics (edge_consistency, temporal) need at least MIN_SAMPLE_SIZE repeated
pairs / gaps on a side; with fewer they count as "absent" (distance 1.0 if the other side has them).

Distances: KS for numeric distributions, total variation for categorical,
absolute difference for rates, |log ratio| for unbounded counts.
"""
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
REAL_TRANS_PATH = "real_trans.csv"
SYNTHETIC_TRANS_PATH = "synthetic_trans.csv"
OUT_PATH = "graph_fidelity_results.csv"

# Maps the names used in this module to the column names of your tables.
COLUMN_MAP = dict(
    account_id="account_id",
    date="trans_date",
    amount="amount",
    trans_type="trans_type",
    operation="operation",
    k_symbol="k_symbol",
    counterparty_bank="bank",
    counterparty_account="account",
    # "days": numeric day offsets (ClavaDDPM-encoded dates; floats are floored).
    # "%y%m%d": Berka YYMMDD integers. None: datetime column or ISO string.
    date_format="days",
    # Values of the counterparty account column that mean "no counterparty".
    counterparty_missing_values=("0",),
)

MISSING_STRINGS = {"", "nan", "none", "<na>", "null", "nat"}
LOG_RATIO_EPSILON = 1e-6        # added to both sides of a log ratio to avoid log(0)
MIN_SAMPLE_SIZE = 30            # minimum rows / repeated pairs / gaps for a comparison
MIN_NOISE_FLOOR = 1e-3          # noise floors below this are clipped when forming the ratio
GAP_RATIO_THRESHOLD = 2         # ratio above this is flagged "GAP"

# Account-level features fed to the classifier two-sample test.
CLASSIFIER_FEATURES = [
    "n_transactions", "n_counterparty_transactions", "counterparty_share",
    "n_distinct_counterparties", "repeat_rate", "max_pair_multiplicity",
    "counterparty_entropy", "mean_counterparty_indegree", "frac_shared_counterparties",
    "n_banks", "bank_entropy", "median_gap_days",
]

# Output metric name -> account-feature column (the "account" metric group).
ACCOUNT_METRIC_COLUMNS = {
    "n_distinct_cp": "n_distinct_counterparties",
    "repeat_rate": "repeat_rate",
    "cp_entropy": "counterparty_entropy",
    "frac_shared_cp": "frac_shared_counterparties",
}

# Output labels -> internal column names for the conditional ("coupling") metrics.
PRESENCE_CATEGORIES = {"type": "trans_type", "operation": "operation", "k_symbol": "k_symbol"}
COUPLING_BUCKETINGS = {"edge_mult": "pair_multiplicity", "cp_indeg": "counterparty_indegree"}
COUPLING_VARIABLES = {
    "log_amt": "log_amount", "type": "trans_type", "operation": "operation",
    "k_symbol": "k_symbol", "bank": "counterparty_bank",
}


# --------------------------------------------------------------------------- #
# Preparation
# --------------------------------------------------------------------------- #
def _clean_id_column(series):
    """String ids with missing values (NaN, 'nan', '', ...) mapped to NaN."""
    series = series.astype("object")
    is_na = series.isna()
    cleaned = (series.where(~is_na, "nan").astype(str).str.strip()
               .str.replace(r"\.0+$", "", regex=True))
    is_missing = is_na | cleaned.str.lower().isin(MISSING_STRINGS)
    return cleaned.astype("object").mask(is_missing)


def _parse_dates(series, date_format):
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    if date_format == "days":
        days = np.floor(pd.to_numeric(series, errors="coerce"))
        return pd.Timestamp("2000-01-01") + pd.to_timedelta(days, unit="D")
    if date_format is None:
        return pd.to_datetime(series, errors="coerce")
    return pd.to_datetime(_clean_id_column(series), format=date_format, errors="coerce")


def clean_transactions(trans, column_map):
    """Standardised copy of the transaction table with internal column names."""
    out = pd.DataFrame(index=trans.index)
    out["account"] = _clean_id_column(trans[column_map["account_id"]])

    counterparty_account = _clean_id_column(trans[column_map["counterparty_account"]])
    counterparty_account = counterparty_account.mask(
        counterparty_account.isin(column_map.get("counterparty_missing_values", ())))
    bank = _clean_id_column(trans[column_map["counterparty_bank"]])

    out["counterparty_account_number"] = counterparty_account
    out["has_counterparty"] = counterparty_account.notna()
    out["counterparty_bank"] = bank.fillna("?").where(out["has_counterparty"])
    out["counterparty"] = (bank.fillna("?") + ":" + counterparty_account).where(
        out["has_counterparty"])

    amount = pd.to_numeric(trans[column_map["amount"]], errors="coerce")
    out["log_amount"] = np.log1p(amount.abs())
    for name in ("trans_type", "operation", "k_symbol"):
        out[name] = _clean_id_column(trans[column_map[name]]).fillna("NA")
    out["date"] = _parse_dates(trans[column_map["date"]], column_map["date_format"])
    return out[out["account"].notna()].copy()


def match_account_counts(real_trans, synthetic_trans, column_map, seed=0):
    """Subsample accounts of the larger table when the account counts differ by > 5%."""
    n_real = _clean_id_column(real_trans[column_map["account_id"]]).nunique()
    n_synthetic = _clean_id_column(synthetic_trans[column_map["account_id"]]).nunique()
    if abs(n_real - n_synthetic) / max(n_real, n_synthetic) <= 0.05:
        return real_trans, synthetic_trans
    n_keep = min(n_real, n_synthetic)
    rng = np.random.default_rng(seed)

    def subsample_accounts(trans):
        account_ids = _clean_id_column(trans[column_map["account_id"]])
        kept = rng.choice(account_ids.dropna().unique(), size=n_keep, replace=False)
        return trans[account_ids.isin(set(kept))]

    if n_real > n_synthetic:
        return subsample_accounts(real_trans), synthetic_trans
    return real_trans, subsample_accounts(synthetic_trans)


def split_accounts_in_half(trans, column_map, seed):
    """Random split of the accounts into two disjoint halves (for the noise floor)."""
    account_ids = _clean_id_column(trans[column_map["account_id"]])
    unique_accounts = account_ids.dropna().unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_accounts)
    first_half = set(unique_accounts[: len(unique_accounts) // 2])
    in_first_half = account_ids.isin(first_half)
    return trans[in_first_half], trans[~in_first_half & account_ids.notna()]


# --------------------------------------------------------------------------- #
# Graph helpers
# --------------------------------------------------------------------------- #
def bipartite_adjacency(account_idx, counterparty_idx, n_accounts, n_counterparties):
    """Accounts x counterparties sparse matrix (entry = 1 per distinct pair)."""
    return sparse.csr_matrix(
        (np.ones(len(account_idx), dtype=np.int64), (account_idx, counterparty_idx)),
        shape=(n_accounts, n_counterparties),
    )


def connected_component_stats(adjacency):
    n_accounts, n_counterparties = adjacency.shape
    full_graph = sparse.bmat([[None, adjacency], [adjacency.T, None]], format="csr")
    _, node_component = connected_components(full_graph, directed=False)
    component_sizes = np.bincount(node_component)
    return dict(
        giant_component_fraction=component_sizes.max() / (n_accounts + n_counterparties),
        dyad_component_share=float((component_sizes == 2).mean()),
    )


def count_butterflies_and_wedges(adjacency):
    """Number of 4-cycles (K_{2,2}) and of wedges centred on counterparties."""
    adjacency = adjacency.tocsr()
    counterparty_degree = np.asarray(adjacency.sum(0)).ravel()
    account_degree = np.asarray(adjacency.sum(1)).ravel()
    # account pairs sharing a counterparty
    wedges_via_counterparties = float(
        (counterparty_degree * (counterparty_degree - 1) / 2).sum())
    wedges_via_accounts = float((account_degree * (account_degree - 1) / 2).sum())
    # count shared neighbours on the side that has fewer wedges (cheaper product)
    if wedges_via_counterparties <= wedges_via_accounts:
        shared_neighbours = adjacency @ adjacency.T
    else:
        shared_neighbours = adjacency.T @ adjacency
    shared_neighbours = sparse.triu(shared_neighbours, k=1)
    pair_counts = shared_neighbours.data.astype(np.float64)
    n_butterflies = float((pair_counts * (pair_counts - 1) / 2).sum())
    return n_butterflies, wedges_via_counterparties


def degree_preserving_edge_swaps(account_idx, counterparty_idx, n_swaps, rng):
    """Randomise a simple bipartite graph with double edge swaps (degrees preserved)."""
    n_counterparties = int(counterparty_idx.max()) + 1
    accounts = account_idx.tolist()
    counterparties = counterparty_idx.tolist()
    existing_edges = {a * n_counterparties + c for a, c in zip(accounts, counterparties)}
    candidate_pairs = rng.integers(0, len(accounts), size=(n_swaps, 2)).tolist()
    for i, j in candidate_pairs:
        account_i, counterparty_i = accounts[i], counterparties[i]
        account_j, counterparty_j = accounts[j], counterparties[j]
        if account_i == account_j or counterparty_i == counterparty_j:
            continue
        new_edge_i = account_i * n_counterparties + counterparty_j
        new_edge_j = account_j * n_counterparties + counterparty_i
        if new_edge_i in existing_edges or new_edge_j in existing_edges:
            continue
        existing_edges.remove(account_i * n_counterparties + counterparty_i)
        existing_edges.remove(account_j * n_counterparties + counterparty_j)
        existing_edges.add(new_edge_i)
        existing_edges.add(new_edge_j)
        counterparties[i], counterparties[j] = counterparty_j, counterparty_i
    return np.asarray(counterparties, dtype=np.int64)


# --------------------------------------------------------------------------- #
# Per-dataset bundle of metrics
# --------------------------------------------------------------------------- #
def build_metric_bundle(trans, column_map):
    """Everything needed to compare one table: cleaned data, per-account features and
    the single-table metrics (`bundle["metrics"]`: (group, name) -> (kind, payload))."""
    transactions = clean_transactions(trans, column_map)
    edges = (transactions[transactions["has_counterparty"]]
             .sort_values("date", kind="stable").copy())
    if edges.empty:
        raise ValueError("No transactions with a counterparty.")
    pair_keys = ["account", "counterparty"]

    edges["pair_multiplicity"] = edges.groupby(pair_keys, sort=False)["counterparty"].transform("size")
    counterparty_stats = edges.groupby("counterparty").agg(
        n_transactions=("counterparty", "size"), indegree=("account", "nunique"))
    edges["counterparty_indegree"] = edges["counterparty"].map(counterparty_stats["indegree"])
    edges["is_repeat_payment"] = edges.groupby(pair_keys, sort=False).cumcount() > 0
    edges["days_since_previous_payment"] = (
        edges.groupby(pair_keys, sort=False)["date"].diff().dt.days)

    pairs = edges.groupby(pair_keys, sort=False).size().rename("n_payments").reset_index()
    pairs = pairs.merge(counterparty_stats["indegree"], left_on="counterparty", right_index=True)
    pairs["counterparty_is_shared"] = pairs["indegree"] > 1
    payment_share = pairs["n_payments"] / pairs.groupby("account")["n_payments"].transform("sum")
    pairs["entropy_term"] = -payment_share * np.log(payment_share)

    # account feature table ---------------------------------------------------
    accounts = pd.DataFrame(index=pd.Index(transactions["account"].unique(), name="account"))
    accounts["n_transactions"] = transactions.groupby("account").size()
    accounts["n_counterparty_transactions"] = (
        edges.groupby("account").size().reindex(accounts.index).fillna(0))
    accounts["counterparty_share"] = (
        accounts["n_counterparty_transactions"] / accounts["n_transactions"])
    pairs_by_account = pairs.groupby("account")
    accounts["n_distinct_counterparties"] = pairs_by_account.size()
    accounts["max_pair_multiplicity"] = pairs_by_account["n_payments"].max()
    accounts["counterparty_entropy"] = pairs_by_account["entropy_term"].sum()
    accounts["mean_counterparty_indegree"] = pairs_by_account["indegree"].mean()
    accounts["frac_shared_counterparties"] = pairs_by_account["counterparty_is_shared"].mean()
    accounts["repeat_rate"] = 1 - accounts["n_distinct_counterparties"] / accounts[
        "n_counterparty_transactions"].where(accounts["n_counterparty_transactions"] > 0)

    account_bank_counts = (edges.groupby(["account", "counterparty_bank"]).size()
                           .rename("n_transactions").reset_index())
    account_bank_counts["share"] = account_bank_counts["n_transactions"] / account_bank_counts.groupby(
        "account")["n_transactions"].transform("sum")
    account_bank_counts["entropy_term"] = -account_bank_counts["share"] * np.log(
        account_bank_counts["share"])
    accounts["n_banks"] = account_bank_counts.groupby("account").size()
    accounts["bank_entropy"] = account_bank_counts.groupby("account")["entropy_term"].sum()
    accounts["median_gap_days"] = edges.groupby("account")["days_since_previous_payment"].median()
    accounts_with_counterparty = accounts[accounts["n_counterparty_transactions"] > 0]

    # bipartite account-counterparty graph ------------------------------------
    account_idx, account_labels = pd.factorize(pairs["account"])
    counterparty_idx, counterparty_labels = pd.factorize(pairs["counterparty"])
    n_accounts, n_counterparties = len(account_labels), len(counterparty_labels)
    adjacency = bipartite_adjacency(account_idx, counterparty_idx, n_accounts, n_counterparties)
    components = connected_component_stats(adjacency)
    n_butterflies, n_wedges = count_butterflies_and_wedges(adjacency)
    n_pairs = len(pairs)

    # bank co-usage (account x bank) ------------------------------------------
    bank_codes, bank_labels = pd.factorize(account_bank_counts["counterparty_bank"])
    account_codes, _ = pd.factorize(account_bank_counts["account"])
    account_bank_matrix = sparse.csr_matrix(
        (np.ones(len(account_bank_counts)), (account_codes, bank_codes)))
    bank_cousage = pd.DataFrame(
        (account_bank_matrix.T @ account_bank_matrix).toarray() / account_bank_matrix.shape[0],
        index=bank_labels, columns=bank_labels)

    # temporal ------------------------------------------------------------------
    gaps = edges["days_since_previous_payment"].dropna()
    pair_gap_stats = (edges.dropna(subset=["days_since_previous_payment"])
                      .groupby(pair_keys)["days_since_previous_payment"]
                      .agg(["count", "mean", "std"]))
    pair_gap_stats = pair_gap_stats[(pair_gap_stats["count"] >= 2) & (pair_gap_stats["mean"] > 0)]
    gap_variation_per_pair = (pair_gap_stats["std"] / pair_gap_stats["mean"]).dropna()
    nearest_month = np.round(gaps / 30.0)
    monthly_gap_share = (
        float(((nearest_month >= 1) & ((gaps - 30 * nearest_month).abs() <= 3)).mean())
        if len(gaps) >= MIN_SAMPLE_SIZE else np.nan)

    # recurring-payment consistency: attributes should stay stable within a repeated pair
    repeated_pairs = edges.groupby(pair_keys, sort=False).agg(
        n_payments=("counterparty", "size"),
        n_distinct_k_symbols=("k_symbol", "nunique"),
        n_distinct_operations=("operation", "nunique"),
        n_distinct_types=("trans_type", "nunique"),
        log_amount_sd=("log_amount", "std"))
    repeated_pairs = repeated_pairs[repeated_pairs["n_payments"] >= 2]
    enough_repeated_pairs = len(repeated_pairs) >= MIN_SAMPLE_SIZE

    def share_with_single_value(column):
        return float((repeated_pairs[column] == 1).mean()) if enough_repeated_pairs else np.nan

    # a counterparty account number should belong to a single bank
    multi_bank_share = float(
        (edges.groupby("counterparty_account_number")["counterparty_bank"].nunique() > 1).mean())

    counterparty_volume = counterparty_stats["n_transactions"].sort_values(ascending=False)
    n_top_counterparties = max(1, int(round(0.01 * len(counterparty_volume))))

    metrics = {}

    def register(group, name, kind, payload):
        metrics[(group, name)] = (kind, payload)

    register("presence", "cp_presence_rate", "rate", float(transactions["has_counterparty"].mean()))
    register("presence", "accounts_with_any_cp", "rate",
             float((accounts["n_counterparty_transactions"] > 0).mean()))
    register("presence", "cp_share_per_account", "ks", accounts["counterparty_share"].values)

    register("account", "n_tx", "ks", accounts["n_transactions"].values)
    register("account", "n_cp_tx", "ks", accounts["n_counterparty_transactions"].values)
    for metric_name, column in ACCOUNT_METRIC_COLUMNS.items():
        register("account", metric_name, "ks", accounts_with_counterparty[column].dropna().values)

    register("pair", "edge_multiplicity", "ks", pairs["n_payments"].values)
    register("pair", "repeat_tx_share", "rate", float(edges["is_repeat_payment"].mean()))

    register("counterparty", "cp_indegree", "ks", counterparty_stats["indegree"].values)
    register("counterparty", "cp_tx_count", "ks", counterparty_stats["n_transactions"].values)
    register("counterparty", "tx_to_shared_cp_share", "rate",
             float((edges["counterparty_indegree"] > 1).mean()))
    register("counterparty", "top1pct_cp_tx_share", "rate",
             float(counterparty_volume.iloc[:n_top_counterparties].sum() / counterparty_volume.sum()))
    register("counterparty", "acct_number_multi_bank_share", "rate", multi_bank_share)

    register("edge_consistency", "repeated_edge_k_symbol_purity", "rate",
             share_with_single_value("n_distinct_k_symbols"))
    register("edge_consistency", "repeated_edge_operation_purity", "rate",
             share_with_single_value("n_distinct_operations"))
    register("edge_consistency", "repeated_edge_type_purity", "rate",
             share_with_single_value("n_distinct_types"))
    register("edge_consistency", "repeated_edge_log_amount_sd", "ks",
             repeated_pairs["log_amount_sd"].dropna().values if enough_repeated_pairs else np.array([]))

    register("structure", "giant_component_frac", "rate", components["giant_component_fraction"])
    register("structure", "dyad_component_share", "rate", components["dyad_component_share"])
    register("structure", "wedges_per_edge", "logratio", n_wedges / n_pairs)
    register("structure", "butterflies_per_edge", "logratio", n_butterflies / n_pairs)

    register("bank", "bank_mix", "tv", edges["counterparty_bank"].value_counts())
    register("bank", "banks_per_account", "ks", accounts_with_counterparty["n_banks"].dropna().values)
    register("bank", "bank_cousage_matrix", "matrix", bank_cousage)

    register("temporal", "gap_days", "ks", gaps.values if len(gaps) >= MIN_SAMPLE_SIZE else np.array([]))
    register("temporal", "gap_cv_per_pair", "ks",
             gap_variation_per_pair.values if len(gap_variation_per_pair) >= MIN_SAMPLE_SIZE
             else np.array([]))
    register("temporal", "monthly_gap_share", "rate", monthly_gap_share)
    register("temporal", "date_parse_fail_rate", "rate", float(transactions["date"].isna().mean()))

    return dict(
        transactions=transactions,
        edges=edges,
        accounts=accounts,
        metrics=metrics,
        graph_indices=(account_idx, counterparty_idx, n_accounts, n_counterparties),
        n_butterflies=n_butterflies,
        n_pairs=n_pairs,
        shared_counterparty_share=float((counterparty_stats["indegree"] > 1).mean()),
    )


# --------------------------------------------------------------------------- #
# Distances
# --------------------------------------------------------------------------- #
DISTANCE_DESCRIPTIONS = {
    "ks": (
        "KS statistic",
        "Largest vertical gap between the two cumulative distribution curves. "
        "Compares the whole distribution, not just the mean. 0 = identical, 1 = no overlap."),
    "tv": (
        "Total variation",
        "Half the sum of absolute differences between category shares (the share of "
        "probability mass that would have to move). Compares the whole categorical "
        "distribution. 0 = identical, 1 = disjoint."),
    "rate": (
        "Absolute difference",
        "abs(real - synthetic) of one number (a rate, share or score). The metric is already "
        "a single value, so there is no distribution to compare. 0 = identical."),
    "logratio": (
        "Absolute log-ratio",
        "abs(log((real + eps) / (synthetic + eps))) for an unbounded non-negative count. "
        "Scale-free: 0 = equal, 0.69 = a factor of 2 apart."),
    "matrix": (
        "Mean absolute matrix difference",
        "Mean over all cells of abs(real - synthetic) for the bank co-usage matrix "
        "(share of accounts using both bank i and bank j). Compares the full pairwise "
        "structure cell by cell. 0 = identical."),
    "wabs": (
        "Weighted absolute rate difference",
        "Per category, abs(real rate - synthetic rate) of having a counterparty, averaged with "
        "weights = real category size. For a yes/no variable the rate is its whole "
        "distribution, so each category is compared exactly. 0 = identical."),
    "weighted_ks": (
        "KS statistic per bucket, weighted",
        "KS statistic within each graph-position bucket, averaged with weights = real bucket "
        "size. A bucket that exists in real but is (almost) empty in synthetic counts as 1."),
    "weighted_tv": (
        "Total variation per bucket, weighted",
        "Total variation within each graph-position bucket, averaged with weights = real bucket "
        "size. A bucket that exists in real but is (almost) empty in synthetic counts as 1."),
    "auc": (
        "Classifier AUC distance",
        "abs(AUC - 0.5) of a gradient-boosted classifier separating real from synthetic accounts "
        "using all per-account graph features jointly (so joint distributions, not just "
        "marginals, matter). 0 = indistinguishable, 0.5 = perfectly separable."),
}


def _total_variation(counts_a, counts_b):
    """Total variation distance between two category-count Series."""
    p = counts_a / counts_a.sum()
    q = counts_b / counts_b.sum()
    categories = p.index.union(q.index)
    return 0.5 * float(
        (p.reindex(categories).fillna(0) - q.reindex(categories).fillna(0)).abs().sum())


def _distance(kind, real_payload, synthetic_payload):
    if kind == "ks":
        if len(real_payload) == 0 and len(synthetic_payload) == 0:
            return 0.0
        if len(real_payload) == 0 or len(synthetic_payload) == 0:
            return 1.0  # quantity exists on only one side
        return float(ks_2samp(real_payload, synthetic_payload).statistic)
    if kind == "tv":
        return _total_variation(real_payload, synthetic_payload)
    if kind in ("rate", "logratio"):
        real_missing, synthetic_missing = np.isnan(real_payload), np.isnan(synthetic_payload)
        if real_missing and synthetic_missing:
            return 0.0
        if real_missing or synthetic_missing:
            return 1.0  # quantity exists on only one side
        if kind == "rate":
            return abs(real_payload - synthetic_payload)
        return abs(float(np.log((real_payload + LOG_RATIO_EPSILON)
                                / (synthetic_payload + LOG_RATIO_EPSILON))))
    if kind == "matrix":
        labels = real_payload.index.union(synthetic_payload.index)
        real_values = real_payload.reindex(index=labels, columns=labels).fillna(0).values
        synthetic_values = synthetic_payload.reindex(index=labels, columns=labels).fillna(0).values
        return float(np.abs(real_values - synthetic_values).mean())
    raise ValueError(kind)


def _summary_value(kind, payload):
    """Scalar shown in the `real` / `synthetic` columns (mean for distributions)."""
    if kind == "ks":
        return float(np.mean(payload)) if len(payload) else np.nan
    if kind in ("rate", "logratio"):
        return float(payload)
    return np.nan


def compare_single_table_metrics(real_bundle, synthetic_bundle):
    """Rows (group, metric, kind, real, synthetic, distance) for metrics computed per table."""
    rows = []
    for (group, name), (kind, real_payload) in real_bundle["metrics"].items():
        synthetic_payload = synthetic_bundle["metrics"][(group, name)][1]
        rows.append((group, name, kind,
                     _summary_value(kind, real_payload), _summary_value(kind, synthetic_payload),
                     _distance(kind, real_payload, synthetic_payload)))
    return rows


def _assign_bucket(edges, bucketing):
    if bucketing == "pair_multiplicity":
        return pd.cut(edges["pair_multiplicity"], [0, 1, 3, np.inf],
                      labels=["1", "2-3", "4+"]).astype(str)
    return pd.cut(edges["counterparty_indegree"], [0, 1, np.inf], labels=["1", "2+"]).astype(str)


def compare_conditional_metrics(real_bundle, synthetic_bundle):
    """Metrics that need both tables at once (presence by category, coupling)."""
    rows = []
    real_transactions = real_bundle["transactions"]
    synthetic_transactions = synthetic_bundle["transactions"]

    for label, column in PRESENCE_CATEGORIES.items():
        real_stats = real_transactions.groupby(column)["has_counterparty"].agg(["mean", "size"])
        synthetic_rate = (synthetic_transactions.groupby(column)["has_counterparty"].mean()
                          .reindex(real_stats.index).fillna(0))
        weights = real_stats["size"] / real_stats["size"].sum()
        rows.append(("presence", f"cp_presence_by_{label}", "wabs", np.nan, np.nan,
                     float((weights * (real_stats["mean"] - synthetic_rate).abs()).sum())))

    real_edges, synthetic_edges = real_bundle["edges"], synthetic_bundle["edges"]
    for bucketing_label, bucketing in COUPLING_BUCKETINGS.items():
        real_bucket = _assign_bucket(real_edges, bucketing)
        synthetic_bucket = _assign_bucket(synthetic_edges, bucketing)
        for variable_label, variable in COUPLING_VARIABLES.items():
            weighted_distance_sum, total_weight = 0.0, 0.0
            for bucket, n_real_rows in real_bucket.value_counts().items():
                real_rows = real_edges[real_bucket == bucket]
                synthetic_rows = synthetic_edges[synthetic_bucket == bucket]
                if len(real_rows) < MIN_SAMPLE_SIZE:
                    continue
                if len(synthetic_rows) < MIN_SAMPLE_SIZE:
                    distance = 1.0  # bucket exists in real data but (almost) not in synthetic
                elif variable == "log_amount":
                    distance = ks_2samp(real_rows[variable].dropna(),
                                        synthetic_rows[variable].dropna()).statistic
                else:
                    distance = _total_variation(real_rows[variable].value_counts(),
                                                synthetic_rows[variable].value_counts())
                weighted_distance_sum += n_real_rows * distance
                total_weight += n_real_rows
            rows.append(("coupling", f"{variable_label} | {bucketing_label} bucket",
                         "weighted_ks" if variable == "log_amount" else "weighted_tv", np.nan, np.nan,
                         weighted_distance_sum / total_weight if total_weight else np.nan))
    return rows


# --------------------------------------------------------------------------- #
# Null-model calibration and classifier two-sample test
# --------------------------------------------------------------------------- #
def butterfly_null_calibration(bundle, n_null_graphs, swaps_per_edge, rng):
    """Observed butterflies relative to degree-preserving random rewirings."""
    account_idx, counterparty_idx, n_accounts, n_counterparties = bundle["graph_indices"]
    null_counts = []
    for _ in range(n_null_graphs):
        rewired = degree_preserving_edge_swaps(
            account_idx, counterparty_idx, int(swaps_per_edge * len(account_idx)), rng)
        null_adjacency = bipartite_adjacency(account_idx, rewired, n_accounts, n_counterparties)
        null_counts.append(count_butterflies_and_wedges(null_adjacency)[0])
    null_counts = np.asarray(null_counts, dtype=float)
    observed = bundle["n_butterflies"]
    null_sd = null_counts.std(ddof=1) if len(null_counts) > 1 else np.nan
    z_score = (observed - null_counts.mean()) / null_sd if null_sd and null_sd > 0 else np.nan
    excess_ratio = (observed + 1) / (null_counts.mean() + 1)
    return excess_ratio, z_score


def classifier_two_sample_test(features_a, features_b, seed=0, n_folds=5):
    """Cross-validated AUC of a classifier separating the two sets of account features."""
    n = min(len(features_a), len(features_b))
    sample_a = features_a.sample(n, random_state=seed)
    sample_b = features_b.sample(n, random_state=seed)
    X = pd.concat([sample_a, sample_b])[CLASSIFIER_FEATURES]
    X = X.loc[:, X.notna().sum() >= 20]  # drop (almost) empty features
    y = np.r_[np.zeros(n), np.ones(n)]
    classifier = HistGradientBoostingClassifier(max_depth=3, max_iter=100, random_state=seed)
    folds = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
    predicted = cross_val_predict(classifier, X, y, cv=folds, method="predict_proba")[:, 1]
    return float(roc_auc_score(y, predicted))


# --------------------------------------------------------------------------- #
# Orchestration
# --------------------------------------------------------------------------- #
def evaluate(real_trans, synthetic_trans, column_map=COLUMN_MAP, n_null_graphs=3,
             swaps_per_edge=5, n_noise_floor_splits=3, seed=0,
             min_shared_counterparty_share_for_motifs=0.05):
    """Compare `synthetic_trans` to `real_trans`; returns one row per metric with
    columns group, metric, distance_measure, distance_explanation, real, synthetic,
    distance, noise_floor, ratio, flag."""
    rng = np.random.default_rng(seed)
    real_trans, synthetic_trans = match_account_counts(
        real_trans, synthetic_trans, column_map, seed)

    real_bundle = build_metric_bundle(real_trans, column_map)
    synthetic_bundle = build_metric_bundle(synthetic_trans, column_map)
    result_columns = ["group", "metric", "kind", "real", "synthetic", "distance"]
    rows = (compare_single_table_metrics(real_bundle, synthetic_bundle)
            + compare_conditional_metrics(real_bundle, synthetic_bundle))

    # Wedge / butterfly metrics only carry information when counterparties are shared
    # between accounts in the REAL data (otherwise the graph is a forest of stars).
    use_motifs = (real_bundle["shared_counterparty_share"]
                  >= min_shared_counterparty_share_for_motifs)
    if use_motifs:
        real_excess, real_z = butterfly_null_calibration(
            real_bundle, n_null_graphs, swaps_per_edge, rng)
        synthetic_excess, synthetic_z = butterfly_null_calibration(
            synthetic_bundle, n_null_graphs, swaps_per_edge, rng)
        rows.append(("structure", "butterfly_excess_vs_null (obs+1)/(null+1)", "logratio",
                     real_excess, synthetic_excess,
                     _distance("logratio", real_excess, synthetic_excess)))
        rows.append(("structure", "butterfly_z_vs_null", "rate", real_z, synthetic_z,
                     abs(real_z - synthetic_z)
                     if np.isfinite(real_z) and np.isfinite(synthetic_z) else np.nan))

    # Noise floor: the same distances between two random halves of the real accounts.
    noise_floor_frames, noise_floor_aucs = [], []
    for split in range(n_noise_floor_splits):
        half_a, half_b = split_accounts_in_half(real_trans, column_map, seed + split)
        bundle_a = build_metric_bundle(half_a, column_map)
        bundle_b = build_metric_bundle(half_b, column_map)
        noise_floor_frames.append(pd.DataFrame(
            compare_single_table_metrics(bundle_a, bundle_b)
            + compare_conditional_metrics(bundle_a, bundle_b), columns=result_columns))
        noise_floor_aucs.append(classifier_two_sample_test(
            bundle_a["accounts"], bundle_b["accounts"], seed + split))
    noise_floor = (pd.concat(noise_floor_frames).groupby(["group", "metric"])["distance"]
                   .mean().rename("noise_floor"))

    auc_metric_name = "account_graph_features_AUC (0.5 = indistinguishable)"
    auc = classifier_two_sample_test(
        real_bundle["accounts"], synthetic_bundle["accounts"], seed)
    rows.append(("c2st", auc_metric_name, "auc", auc, np.nan, abs(auc - 0.5)))
    noise_floor = pd.concat([noise_floor, pd.Series(
        {("c2st", auc_metric_name): float(np.mean([abs(x - 0.5) for x in noise_floor_aucs]))},
        name="noise_floor")])

    results = pd.DataFrame(rows, columns=result_columns).join(
        noise_floor, on=["group", "metric"])
    results["ratio"] = results["distance"] / results["noise_floor"].clip(lower=MIN_NOISE_FLOOR)
    results["flag"] = np.where(
        results["ratio"].isna(), "",
        np.where(results["ratio"] <= GAP_RATIO_THRESHOLD, "ok", "GAP"))
    results["distance_measure"] = results["kind"].map(
        {kind: name for kind, (name, _) in DISTANCE_DESCRIPTIONS.items()})
    results["distance_explanation"] = results["kind"].map(
        {kind: text for kind, (_, text) in DISTANCE_DESCRIPTIONS.items()})
    results = results.drop(columns="kind")[
        ["group", "metric", "distance_measure", "distance_explanation", "real", "synthetic",
         "distance", "noise_floor", "ratio", "flag"]]
    if not use_motifs:
        results = results[~results["metric"].isin(["wedges_per_edge", "butterflies_per_edge"])]
    group_order = ["presence", "account", "pair", "counterparty", "edge_consistency",
                   "structure", "bank", "temporal", "coupling", "c2st"]
    results["group"] = pd.Categorical(results["group"], group_order, ordered=True)
    return results.sort_values("group", kind="stable").reset_index(drop=True)


In [3]:
import os
from pathlib import Path


def set_project_root(marker="pyproject.toml"):
    """Change the working directory to the repository root."""
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


set_project_root()

PosixPath('/home/coder/vector-bootcamp-synthetic-data')

In [4]:
def drop_aux_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Drop placeholder and cluster columns added during synthesis."""
    drop = [col for col in df.columns if col == "placeholder" or col.endswith("_cluster")]
    return df.drop(columns=drop, errors="ignore")

## Why the comparison is structured this way

The Berka preprocessing chooses complete account transaction groups until it reaches about 20,000 `trans` rows. Sampling whole accounts, rather than individual transactions, keeps each selected account's sibling sequence intact and its foreign keys valid, while limiting the size of this large table. The other tables, including all 4,500 accounts, remain unsampled. The result is 20,284 rows from only 88 active accounts. This makes the training table sparse: most accounts have no transaction children, so ClavaDDPM learns a sparse account-to-transaction count distribution. Its current sample has 14,091 transactions across 63 active accounts. The `Sample size: 20284` log is a reference-size hint for a child table, not a row-count guarantee.

RelDiff was sampled with `--structure original`. Its per-account transaction counts match the real data, so its graph/cardinality scores are conditioned on known structure and should not be read as evidence that RelDiff learned to generate that structure. Those metrics remain useful for comparing attributes and graph features under the available samples, but this is not an end-to-end structure-generation comparison.

For the sibling balance question, the notebook samples 63 active accounts from real and RelDiff to match ClavaDDPM's 63. Account IDs are not matched. It then uses chronological, non-overlapping 20-transaction windows and gives each account one median score per metric. This controls the number of accounts and sequence length, so RelDiff does not get credit just for inheriting real transaction counts. It is a conditional comparison, not a held-out test of generalization.

## Reading the tables

The graph table has grouped headers. `real` is the single shared real-data reference; each metric header then has `ClavaDDPM` and `RelDiff` subcolumns. The first number under a model's `synthetic` header is a scalar summary of that model's metric payload (usually the mean for a distribution). It is **not** the distance. For example, `gap_days`'s RelDiff synthetic summary is the mean gap in days; its KS distance appears separately under `distance`.

`distance` measures the real-vs-synthetic discrepancy: KS statistic for distributions, total variation for categories, absolute difference for rates, absolute log-ratio for unbounded counts, and mean absolute difference for matrices. C2ST reports model-specific AUC under `auc`; the corresponding `distance` is `abs(AUC - 0.5)`. `noise_floor` is the same metric's discrepancy between two real-account halves. `ratio` is `distance / max(noise_floor, 0.001)`, and `flag` is `GAP` when the ratio exceeds 2. Lower distance and ratio are better, but interpret the ratio with the raw distance when the noise floor is zero or very small.

`N/A` means there is no applicable scalar summary or not enough observations to estimate one; it does not automatically mean the metric distance is unavailable. Distribution and matrix metrics have no scalar `real` or `synthetic` value, so compare their `distance`. For temporal gap metrics, at least 30 gaps are required; gap variability also requires at least 30 qualifying repeated pairs. In this run ClavaDDPM has 24 counterparty gaps and only 1 pair with at least two gaps, so its `gap_days`, `monthly_gap_share`, and `gap_cv_per_pair` synthetic summaries are `N/A`. Since real has enough observations, their distances are still calculated as 1.0 (the evaluator's penalty for a quantity present only on the real side). Repeated-edge summaries follow the same 30-pair minimum; ClavaDDPM has 23 repeated pairs, so those summaries are `N/A` too. In contrast, ClavaDDPM's `n_distinct_cp` is a valid metric with a numeric value; the name is not an abbreviation for missing data.

The balance table reports `window_transactions` and `accounts_per_dataset` as sample context. `real_mean` and each model's `synthetic_mean` are means of per-account scores; `distance_ks` compares the full distributions of per-account scores. The balance metrics are lag-1 balance autocorrelation and correlation between transaction amount and the within-account balance change. Lower KS distance and ratio indicate a closer match.

## Metric glossary

Each row below describes the quantity whose real and synthetic distributions or rates are compared.

### Presence

| Metric | Measures |
|---|---|
| `accounts_with_any_cp` | Fraction of accounts with at least one transaction that names a counterparty. |
| `cp_presence_by_k_symbol` | Difference in counterparty-presence rates across `k_symbol` categories, weighted by real category counts. |
| `cp_presence_by_operation` | Difference in counterparty-presence rates across operation categories, weighted by real category counts. |
| `cp_presence_by_type` | Difference in counterparty-presence rates across transaction types, weighted by real category counts. |
| `cp_presence_rate` | Fraction of transactions that name a counterparty. |
| `cp_share_per_account` | Distribution of each account's fraction of transactions that name a counterparty. |

### Account

| Metric | Measures |
|---|---|
| `cp_entropy` | How evenly each account spreads its counterparty payments; higher entropy means less concentration. |
| `frac_shared_cp` | Per-account fraction of counterparties also used by other accounts. |
| `n_cp_tx` | Distribution of counterparty-bearing transactions per account. |
| `n_distinct_cp` | Distribution of distinct counterparties per account. |
| `n_tx` | Distribution of all transactions per account. |
| `repeat_rate` | Per-account fraction of counterparty transactions that revisit an existing account-counterparty pair. |

### Pair and counterparty

| Metric | Measures |
|---|---|
| `edge_multiplicity` | Distribution of transaction counts for each account-counterparty pair. |
| `repeat_tx_share` | Fraction of counterparty-bearing transactions that repeat an earlier account-counterparty pair. |
| `acct_number_multi_bank_share` | Fraction of counterparty account numbers observed under multiple banks. |
| `cp_indegree` | Distribution of distinct paying accounts per counterparty. |
| `cp_tx_count` | Distribution of transactions received by each counterparty. |
| `top1pct_cp_tx_share` | Fraction of counterparty-bearing transactions received by the most-used 1% of counterparties. |
| `tx_to_shared_cp_share` | Fraction of counterparty-bearing transactions sent to counterparties used by at least two accounts. |

### Edge consistency

| Metric | Measures |
|---|---|
| `repeated_edge_k_symbol_purity` | Fraction of repeated account-counterparty pairs that keep the same `k_symbol`. |
| `repeated_edge_log_amount_sd` | Distribution of within-pair standard deviations of log transaction amount. |
| `repeated_edge_operation_purity` | Fraction of repeated pairs that keep the same operation. |
| `repeated_edge_type_purity` | Fraction of repeated pairs that keep the same transaction type. |

### Structure and bank

| Metric | Measures |
|---|---|
| `dyad_component_share` | Fraction of graph connected components consisting of exactly one account and one counterparty. |
| `giant_component_frac` | Fraction of graph nodes in the largest connected component. |
| `bank_cousage_matrix` | Mean absolute difference in the account-by-bank co-usage matrix. |
| `bank_mix` | Total-variation distance between counterparty-bank transaction shares. |
| `banks_per_account` | Distribution of distinct counterparty banks used per account. |

### Temporal

| Metric | Measures |
|---|---|
| `gap_cv_per_pair` | Distribution of coefficient of variation (standard deviation / mean) of gaps for repeated pairs. |
| `gap_days` | Distribution of days between consecutive transactions to the same counterparty within an account. |
| `monthly_gap_share` | Fraction of repeated-pair gaps within three days of a whole-number multiple of 30 days. |

### Conditional coupling

These compare an attribute distribution within buckets defined by pair multiplicity (`edge_mult`) or counterparty indegree (`cp_indeg`). Distances are weighted by real bucket size.

| Metric family | Measures |
|---|---|
| `bank | cp_indeg bucket`, `bank | edge_mult bucket` | Counterparty-bank mix within each graph-position bucket. |
| `k_symbol | cp_indeg bucket`, `k_symbol | edge_mult bucket` | `k_symbol` mix within each graph-position bucket. |
| `log_amt | cp_indeg bucket`, `log_amt | edge_mult bucket` | Log-amount distribution within each graph-position bucket. |
| `operation | cp_indeg bucket`, `operation | edge_mult bucket` | Operation mix within each graph-position bucket. |
| `type | cp_indeg bucket`, `type | edge_mult bucket` | Transaction-type mix within each graph-position bucket. |

### Classifier and balance dependency

| Metric | Measures |
|---|---|
| `account_graph_features_AUC (0.5 = indistinguishable)` | Cross-validated classifier AUC for distinguishing real from synthetic accounts using graph features; 0.5 is indistinguishable and 1.0 is perfectly separable. |
| `balance_change_amount_corr` | Per-account correlation between transaction amount and the change from the previous balance. |
| `lag1_balance_autocorr` | Per-account correlation between a balance and the next chronological balance. |

## Current-run summary and conclusion

The graph comparison reports 30 GAPs for ClavaDDPM and 24 for RelDiff across 42 metrics; in conditional coupling the counts are 5 and 0. RelDiff is closer on many of these metrics, but its original-structure sampling preserves the real graph and transaction-count distribution, so this is not evidence that it learned to generate that structure.

For temporal metrics, RelDiff's synthetic summaries are 211.66 mean gap days, 0.793 mean gap coefficient of variation, and 0.265 monthly-gap share. Their KS distances are 0.840, 0.930, and 0.734, respectively. ClavaDDPM has `N/A` summaries for these metrics because it falls below the observation thresholds above; its distances are 1.0. These are limited-data penalties, not precise estimates of its temporal distributions.

For sibling balance dependency, both models are flagged as gaps. Lag-1 balance-autocorrelation KS distance is 0.984 for ClavaDDPM and 0.968 for RelDiff. Balance-change/amount-correlation KS distance is 0.698 and 0.651, respectively. RelDiff is slightly closer on both, but neither reproduces the real within-account balance dependency well. Overall, RelDiff is closer on some measured features, but neither result supports a general winner; the graph results are structure-conditioned for RelDiff, and the balance results show substantial remaining gaps.

The all-zero `date_parse_fail_rate` sanity check is omitted because it does not distinguish the datasets.

In [5]:
BALANCE_METRICS = {
    "lag1_balance_autocorr": "Correlation between consecutive balances within an account.",
    "balance_change_amount_corr": (
        "Correlation between a transaction's amount and its within-account balance change."
    ),
}
BALANCE_WINDOW_SIZE = 20
BALANCE_NOISE_FLOOR_SPLITS = 30


def _safe_corr(first: np.ndarray, second: np.ndarray) -> float:
    """Return Pearson correlation when enough non-constant paired values exist."""
    valid = np.isfinite(first) & np.isfinite(second)
    first = first[valid]
    second = second[valid]
    if len(first) < 3 or np.std(first) == 0 or np.std(second) == 0:
        return float("nan")
    return float(np.corrcoef(first, second)[0, 1])


def balance_dependency_by_account(
    transactions: pd.DataFrame,
    account_column: str,
    date_column: str,
    window_size: int,
) -> pd.DataFrame:
    """Summarize within-account balance transitions using equal-length windows.

    Non-overlapping windows are ordered chronologically. Each account contributes
    one median score per metric regardless of its total number of transactions.
    """
    sort_columns = [date_column]
    if "trans_id" in transactions.columns:
        sort_columns.append("trans_id")
    records = []
    for account_id, account_rows in transactions.groupby(account_column, sort=False):
        ordered = account_rows.sort_values(sort_columns, kind="stable")
        balances = pd.to_numeric(ordered["balance"], errors="coerce").to_numpy(dtype=float)
        amounts = pd.to_numeric(ordered["amount"], errors="coerce").to_numpy(dtype=float)
        lag_correlations = []
        change_correlations = []

        for start in range(0, len(ordered) - window_size + 1, window_size):
            end = start + window_size
            balance_window = balances[start:end]
            amount_window = amounts[start:end]
            if not np.isfinite(balance_window).all() or not np.isfinite(amount_window).all():
                continue
            lag_correlations.append(_safe_corr(balance_window[:-1], balance_window[1:]))
            change_correlations.append(
                _safe_corr(np.diff(balance_window), amount_window[1:])
            )

        records.append(
            {
                "account_id": account_id,
                "lag1_balance_autocorr": (
                    float(np.nanmedian(lag_correlations))
                    if np.isfinite(lag_correlations).any()
                    else float("nan")
                ),
                "balance_change_amount_corr": (
                    float(np.nanmedian(change_correlations))
                    if np.isfinite(change_correlations).any()
                    else float("nan")
                ),
            }
        )
    return pd.DataFrame(records)


def compare_balance_dependencies(
    real_scores: pd.DataFrame,
    synthetic_scores: dict[str, pd.DataFrame],
    window_size: int,
    seed: int = 0,
) -> pd.DataFrame:
    """Compare per-account balance dependency distributions to real with a noise floor."""
    rows = []
    rng = np.random.default_rng(seed)
    for metric_name in BALANCE_METRICS:
        real_values = real_scores[metric_name].dropna().to_numpy(dtype=float)
        if len(real_values) < 4:
            raise ValueError(f"Not enough real accounts for {metric_name}")

        noise_floor_values = []
        for _ in range(BALANCE_NOISE_FLOOR_SPLITS):
            shuffled = rng.permutation(real_values)
            midpoint = len(shuffled) // 2
            noise_floor_values.append(
                float(ks_2samp(shuffled[:midpoint], shuffled[midpoint:]).statistic)
            )
        noise_floor = float(np.mean(noise_floor_values))

        for model_name, scores in synthetic_scores.items():
            synthetic_values = scores[metric_name].dropna().to_numpy(dtype=float)
            distance = float(ks_2samp(real_values, synthetic_values).statistic)
            ratio = distance / max(noise_floor, MIN_NOISE_FLOOR)
            rows.append(
                {
                    "model": model_name,
                    "metric": metric_name,
                    "description": BALANCE_METRICS[metric_name],
                    "window_transactions": window_size,
                    "real_accounts": int(len(real_values)),
                    "synthetic_accounts": int(len(synthetic_values)),
                    "real_mean": float(np.mean(real_values)),
                    "synthetic_mean": float(np.mean(synthetic_values)),
                    "distance_ks": distance,
                    "noise_floor": noise_floor,
                    "ratio": ratio,
                    "flag": "ok" if ratio <= GAP_RATIO_THRESHOLD else "GAP",
                }
            )
    return pd.DataFrame(rows)


In [ ]:
import pickle

ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
base_data_dir = IMPLEMENTATION_ROOT / "multi_table" / "data" / "berka"
base_output_dir = IMPLEMENTATION_ROOT / "multi_table" / "results"
# reldiff_data_root = ROOT / "RelDiff" / "data" / "synthetic" / "berka_reldiff" / "RelDiff"
# reldiff_sample_dirs = sorted(reldiff_data_root.glob("reldiff_*/sample*"))

reldiff_data_root = IMPLEMENTATION_ROOT / "multi_table" / "results" / "RelDiff_synth_data"
reldiff_sample_dirs = sorted(reldiff_data_root.glob("sample*"))

if not reldiff_sample_dirs:
    raise FileNotFoundError(f"No RelDiff synthetic sample found under {reldiff_data_root}")
reldiff_sample_dir = reldiff_sample_dirs[-1]

PARENT_TABLE = "account"
pickle_path = base_output_dir / "before_matching" / "synthetic_tables.pkl"
with open(pickle_path, "rb") as f:
    synthetic_tables = pickle.load(f)

synthetic_trans_by_model = {
    "ClavaDDPM": drop_aux_columns(synthetic_tables[(PARENT_TABLE, "trans")]["df"]),
    "RelDiff": pd.read_csv(reldiff_sample_dir / "trans.csv"),
}
real_trans = pd.read_csv(base_data_dir / "trans.csv")
account_column = COLUMN_MAP["account_id"]
real_account_count = _clean_id_column(real_trans[account_column]).nunique()
synthetic_account_counts = {
    name: _clean_id_column(frame[account_column]).nunique()
    for name, frame in synthetic_trans_by_model.items()
}
n_comparison_accounts = min(real_account_count, *synthetic_account_counts.values())


def sample_accounts(trans: pd.DataFrame, n_accounts: int, seed: int) -> pd.DataFrame:
    """Keep a deterministic account sample for a fair model-to-model comparison."""
    account_ids = _clean_id_column(trans[account_column])
    unique_accounts = account_ids.dropna().unique()
    if len(unique_accounts) <= n_accounts:
        return trans.copy()
    kept_accounts = np.random.default_rng(seed).choice(
        unique_accounts, size=n_accounts, replace=False
    )
    return trans[account_ids.isin(set(kept_accounts))].copy()


real_comparison_trans = sample_accounts(real_trans, n_comparison_accounts, seed=0)
synthetic_trans_by_model = {
    model_name: sample_accounts(model_trans, n_comparison_accounts, seed=index + 1)
    for index, (model_name, model_trans) in enumerate(synthetic_trans_by_model.items())
}
print(
    f"Using the same {n_comparison_accounts}-account real sample for both models "
    f"(original real count: {real_account_count:,}; synthetic counts: {synthetic_account_counts})"
)

comparison_rows = []
for model_name, model_trans in synthetic_trans_by_model.items():
    print(
        f"Evaluating {model_name}: {len(model_trans):,} transactions, "
        f"{model_trans['account_id'].nunique():,} matched accounts"
    )
    model_results = evaluate(real_comparison_trans, model_trans)
    model_results.insert(0, "model", model_name)
    comparison_rows.append(model_results)

results = pd.concat(comparison_rows, ignore_index=True)
# Every dataset has a zero parse-failure rate, so this sanity check does not distinguish them.
results = results.loc[results["metric"] != "date_parse_fail_rate"].copy()
model_fields = ["synthetic", "distance", "noise_floor", "ratio", "flag"]
model_comparison = results.pivot(
    index=["group", "metric"],
    columns="model",
    values=model_fields,
)
model_comparison.columns = pd.MultiIndex.from_tuples(
    [(field, model) for field, model in model_comparison.columns]
)
real_reference = results.groupby(["group", "metric"], observed=True)["real"].first()
c2st_mask = results["group"].eq("c2st")
c2st_index = pd.MultiIndex.from_frame(results.loc[c2st_mask, ["group", "metric"]].drop_duplicates())
real_reference.loc[c2st_index] = np.nan
real_column = real_reference.to_frame()
real_column.columns = pd.MultiIndex.from_tuples([("real", "")])
auc_comparison = results.loc[c2st_mask].pivot(
    index=["group", "metric"], columns="model", values="real"
)
auc_comparison.columns = pd.MultiIndex.from_tuples(
    [("auc", model) for model in auc_comparison.columns]
)
comparison = pd.concat([real_column, model_comparison, auc_comparison], axis=1)
comparison.columns.names = [None, None]
comparison_display = comparison.astype(object).where(pd.notna(comparison), "N/A")
with pd.option_context(
    "display.max_rows", None,
    "display.width", 220,
    "display.max_colwidth", 70,
    "display.float_format", "{:.4f}".format,
):
    print(comparison_display.to_string())
    print("\nGAP count per model and group:")
    print(
        results.assign(gap=results["flag"].eq("GAP"), n=results["flag"].ne(""))
        .groupby(["model", "group"], observed=True)[["gap", "n"]]
        .sum()
    )

minimum_transaction_count = min(
    int(trans.groupby(account_column).size().min())
    for trans in [real_comparison_trans, *synthetic_trans_by_model.values()]
)
balance_window_size = min(BALANCE_WINDOW_SIZE, minimum_transaction_count)
if balance_window_size < 4:
    raise ValueError(
        "Need at least four transactions per account for balance-dependency evaluation; "
        f"found a minimum of {minimum_transaction_count}."
    )

real_balance_scores = balance_dependency_by_account(
    real_comparison_trans,
    account_column,
    COLUMN_MAP["date"],
    balance_window_size,
)
synthetic_balance_scores = {
    model_name: balance_dependency_by_account(
        model_trans,
        account_column,
        COLUMN_MAP["date"],
        balance_window_size,
    )
    for model_name, model_trans in synthetic_trans_by_model.items()
}
balance_comparison_long = compare_balance_dependencies(
    real_balance_scores,
    synthetic_balance_scores,
    balance_window_size,
)
balance_metadata = balance_comparison_long.groupby("metric", observed=True).agg(
    description=("description", "first"),
    window_transactions=("window_transactions", "first"),
    accounts_per_dataset=("real_accounts", "first"),
)
real_balance_means = balance_comparison_long.groupby("metric", observed=True)["real_mean"].first()
balance_model_comparison = balance_comparison_long.pivot(
    index="metric",
    columns="model",
    values=["synthetic_mean", "distance_ks", "noise_floor", "ratio", "flag"],
)
balance_model_comparison.columns = pd.MultiIndex.from_tuples(
    [(field, model) for field, model in balance_model_comparison.columns]
)
setup_columns = balance_metadata.copy()
setup_columns.columns = pd.MultiIndex.from_tuples(
    [("setup", column) for column in setup_columns.columns]
)
real_balance_column = real_balance_means.to_frame()
real_balance_column.columns = pd.MultiIndex.from_tuples([("real mean", "")])
balance_display = pd.concat(
    [setup_columns, real_balance_column, balance_model_comparison], axis=1
)
balance_display.columns.names = [None, None]
balance_display = balance_display.astype(object).where(pd.notna(balance_display), "N/A")
balance_comparison = (
    balance_metadata.join(real_balance_means.rename("real_mean"))
    .join(
        balance_model_comparison.copy().set_axis(
            [f"{model}_{field}" for field, model in balance_model_comparison.columns], axis=1
        )
    )
    .reset_index()
)
print("\nConditional sibling balance-dependency comparison:")
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 220):
    print(balance_display.to_string())

comparison_output_path = ROOT / "graph_fidelity_model_comparison.csv"
flat_comparison = comparison.copy()
flat_comparison.columns = [
    field if not model else f"{model}_{field}" for field, model in flat_comparison.columns
]
flat_comparison.reset_index().to_csv(comparison_output_path, index=False, na_rep="N/A")
balance_output_path = ROOT / "balance_dependency_comparison.csv"
balance_comparison.to_csv(balance_output_path, index=False, na_rep="N/A")
print(f"\nSaved graph comparison to {comparison_output_path}")
print(f"Saved balance-dependency comparison to {balance_output_path}")

Using the same 63-account real sample for both models (original real count: 88; synthetic counts: {'ClavaDDPM': 63, 'RelDiff': 88})
Evaluating ClavaDDPM: 14,091 transactions, 63 matched accounts
Evaluating RelDiff: 13,432 transactions, 63 matched accounts
                                                                          real synthetic           distance         noise_floor             ratio               flag               auc        
                                                                               ClavaDDPM  RelDiff ClavaDDPM RelDiff   ClavaDDPM RelDiff ClavaDDPM  RelDiff ClavaDDPM RelDiff ClavaDDPM RelDiff
group            metric                                                                                                                                                                       
presence         accounts_with_any_cp                                   0.8413    1.0000   0.7937    0.1587  0.0476      0.0652  0.0652    2.4350   0.7305       GAP      o